# Shabaka Pulse — Financial Settlement Engine

This notebook converts absorbed renewable surplus energy into financial settlement records. It aggregates the total MWh absorbed per facility during a sample peak-winter week, computes the gross value unlocked by avoiding Take-or-Pay grid penalties and preserving natural gas, applies a 50/40/10 waterfall revenue split, and exports the resulting billing credit ledgers to JSON.

## 1. Setup

Importing data processing libraries and type hinting utilities.

In [13]:
import json
from pathlib import Path
from typing import Dict, List, Tuple
import pandas as pd

## 2. Define Financial Constants

Defining the core economic parameters that drive settlement value:
- **Take-or-Pay Tariff**: Fixed cost paid for curtailed renewable energy (EGP/MWh).
- **Thermal Heat Rate**: Efficiency of local CCGT natural gas plants (MMBtu of gas burned per MWh generated).
- **LNG Spot Price**: Global market value of natural gas (USD/MMBtu).
- **Exchange Rate**: EGP to USD conversion.

In [14]:
TARIFF_EGP_PER_MWH = 1400.0
HEAT_RATE_MMBTU_PER_MWH = 7.5
LNG_PRICE_USD_PER_MMBTU = 10.5
EXCHANGE_RATE_EGP_PER_USD = 50.0

constants_df = pd.DataFrame([
    {"Parameter": "Take-or-Pay Tariff", "Value": TARIFF_EGP_PER_MWH, "Unit": "EGP/MWh"},
    {"Parameter": "CCGT Heat Rate", "Value": HEAT_RATE_MMBTU_PER_MWH, "Unit": "MMBtu/MWh"},
    {"Parameter": "LNG Spot Price", "Value": LNG_PRICE_USD_PER_MMBTU, "Unit": "USD/MMBtu"},
    {"Parameter": "Exchange Rate", "Value": EXCHANGE_RATE_EGP_PER_USD, "Unit": "EGP/USD"},
])
display(constants_df)

,Parameter,Value,Unit
0,Take-or-Pay Tariff,1400.0,EGP/MWh
1,CCGT Heat Rate,7.5,MMBtu/MWh
2,LNG Spot Price,10.5,USD/MMBtu
3,Exchange Rate,50.0,EGP/USD


## 3. Dispatch Simulation

Loading the hourly surplus predictions and clustered facilities. We extract a peak winter week (January 1 to January 7) and run the priority dispatch solver hour by hour, accumulating the total MWh absorbed by each facility across the period.

In [15]:
def allocate_surplus(
    surplus_mw: float,
    clustered_facilities: pd.DataFrame,
) -> Tuple[Dict[str, float], float]:
    """Allocates renewable surplus across facilities in Tier 1 -> 2 -> 3 priority order."""
    remaining_surplus = float(surplus_mw)
    allocations: Dict[str, float] = {str(fid): 0.0 for fid in clustered_facilities["facility_id"]}
    for tier_num in [1, 2, 3]:
        if remaining_surplus <= 0.0:
            break
        tier_df = clustered_facilities[clustered_facilities["tier"] == tier_num].sort_values("facility_id")
        for _, row in tier_df.iterrows():
            if remaining_surplus <= 0.0:
                break
            fac_id = str(row["facility_id"])
            allocated = min(float(row["max_flex_mw"]), remaining_surplus)
            allocations[fac_id] = round(allocated, 4)
            remaining_surplus -= allocated
    return allocations, round(remaining_surplus, 6)


# Load datasets
surplus_df = pd.read_csv("data/surplus_forecast.csv", index_col="timestamp_utc", parse_dates=True)
facilities_df = pd.read_csv("data/facilities_clustered.csv")

# Extract peak winter sample week (Jan 1 to Jan 7)
sample_week = surplus_df.loc["2025-01-01":"2025-01-07"]

# Accumulate absorbed MWh per facility across all hours in the sample week.
# Each hourly timestep: MW absorbed for 1 hour equals MWh directly.
total_mwh_absorbed: Dict[str, float] = {str(fid): 0.0 for fid in facilities_df["facility_id"]}

for _, hour_row in sample_week.iterrows():
    surplus_val = float(hour_row["surplus_mw"])
    if surplus_val > 0:
        hourly_alloc, _ = allocate_surplus(surplus_val, facilities_df)
        for fac_id, mw in hourly_alloc.items():
            total_mwh_absorbed[fac_id] += mw

absorbed_df = pd.DataFrame(
    list(total_mwh_absorbed.items()), columns=["facility_id", "total_mwh_absorbed"]
).sort_values("total_mwh_absorbed", ascending=False)

total_system = absorbed_df["total_mwh_absorbed"].sum()
print("Total system MWh absorbed during sample week: " + str(round(total_system, 2)) + " MWh")
display(absorbed_df.head())

Total system MWh absorbed during sample week: 392.23 MWh


,facility_id,total_mwh_absorbed
6,FAC-007,102.5700
0,FAC-001,62.9100
9,FAC-010,59.0700
12,FAC-013,50.1389
18,FAC-019,34.7544


## 4. Settlement Calculations

Computing the two distinct value streams unlocked by absorbing renewable surplus:
1. **Avoided Take-or-Pay Cost**: Direct savings from not paying curtailment penalties.
2. **Preserved Gas Value**: Natural gas not burned at thermal plants, priced at LNG spot rates in EGP.

In [16]:
absorbed_df["avoided_top_cost_egp"] = absorbed_df["total_mwh_absorbed"] * TARIFF_EGP_PER_MWH

# Convert gas savings from USD to EGP using the assumed exchange rate
absorbed_df["preserved_gas_usd"] = (
    absorbed_df["total_mwh_absorbed"] * HEAT_RATE_MMBTU_PER_MWH * LNG_PRICE_USD_PER_MMBTU
)
absorbed_df["preserved_gas_egp"] = absorbed_df["preserved_gas_usd"] * EXCHANGE_RATE_EGP_PER_USD

absorbed_df["gross_savings_egp"] = absorbed_df["avoided_top_cost_egp"] + absorbed_df["preserved_gas_egp"]

display(
    absorbed_df[["facility_id", "avoided_top_cost_egp", "preserved_gas_egp", "gross_savings_egp"]]
    .round(2)
    .head()
)

,facility_id,avoided_top_cost_egp,preserved_gas_egp,gross_savings_egp
6,FAC-007,143598.00,403869.38,547467.38
0,FAC-001,88074.00,247708.12,335782.12
9,FAC-010,82698.00,232588.13,315286.13
12,FAC-013,70194.46,197421.92,267616.38
18,FAC-019,48656.16,136845.45,185501.61


## 5. Waterfall Split

Applying the 50/40/10 waterfall split to gross savings:
- **50% Factory Discount**: Billing credit passed to the participating industrial facility.
- **40% Treasury**: Retained by the Ministry of Electricity / Grid Operator.
- **10% Platform Fee**: Shabaka Pulse operational margin.

In [17]:
absorbed_df["factory_credit_egp"] = absorbed_df["gross_savings_egp"] * 0.50
absorbed_df["treasury_retained_egp"] = absorbed_df["gross_savings_egp"] * 0.40
absorbed_df["platform_fee_egp"] = absorbed_df["gross_savings_egp"] * 0.10

split_summary = pd.DataFrame({
    "Total Factory Credits (EGP)": [absorbed_df["factory_credit_egp"].sum()],
    "Total Treasury Retained (EGP)": [absorbed_df["treasury_retained_egp"].sum()],
    "Total Platform Fee (EGP)": [absorbed_df["platform_fee_egp"].sum()],
}).round(2)

display(split_summary)
display(
    absorbed_df[["facility_id", "factory_credit_egp", "treasury_retained_egp", "platform_fee_egp"]]
    .round(2)
    .head()
)

,Total Factory Credits (EGP),Total Treasury Retained (EGP),Total Platform Fee (EGP)
0,1046767.02,837413.61,209353.4


,facility_id,factory_credit_egp,treasury_retained_egp,platform_fee_egp
6,FAC-007,273733.69,218986.95,54746.74
0,FAC-001,167891.06,134312.85,33578.21
9,FAC-010,157643.06,126114.45,31528.61
12,FAC-013,133808.19,107046.55,26761.64
18,FAC-019,92750.80,74200.64,18550.16


## 6. Ledger Generation

Exporting the final financial records as JSON ledgers for each facility. These ledgers serve as the authoritative settlement payload for downstream billing systems.

In [18]:
ledgers: List[Dict] = []

for _, row in absorbed_df.iterrows():
    ledger_entry = {
        "facility_id": str(row["facility_id"]),
        "period": "2025-01-01 to 2025-01-07",
        "mwh_absorbed": round(float(row["total_mwh_absorbed"]), 2),
        "gross_savings_egp": round(float(row["gross_savings_egp"]), 2),
        "factory_discount_credit_egp": round(float(row["factory_credit_egp"]), 2),
    }
    ledgers.append(ledger_entry)

output_path = Path("data/settlement_ledgers.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(ledgers, f, indent=4)

print("Written " + str(len(ledgers)) + " ledger records to " + str(output_path.resolve()))
print(json.dumps(ledgers[0], indent=4))

Written 30 ledger records to D:\shabaka-pluse\data\settlement_ledgers.json
{
    "facility_id": "FAC-007",
    "period": "2025-01-01 to 2025-01-07",
    "mwh_absorbed": 102.57,
    "gross_savings_egp": 547467.38,
    "factory_discount_credit_egp": 273733.69
}
